<a href="https://raw.githubusercontent.com/tsg-/oneccl-tutorial/master/notebooks/04_inference_tp_decode.ipynb" download>⬇️ Download this notebook</a>

# End-to-End: TP Decode Loop

This notebook assembles all the pieces into a complete autoregressive decode loop with
Tensor Parallelism. It is the closest thing to what a production inference server does
during the decode phase.

## What Tensor Parallelism means for decode

In TP=4 inference, the model is split across 4 GPUs along the weight matrix columns/rows:

```
Linear(hidden=8192, ffn=28672) in Llama-3 70B:

  Full weight:  [8192, 28672]   ← too large for one GPU
  TP=4 split:   each GPU holds [8192, 7168]  ← column shard

Forward pass:
  1. Each GPU computes matmul(input, weight_shard) → partial output [batch, seq, 7168]
  2. Allreduce (SUM) across 4 GPUs → full output [batch, seq, 28672]
     (Wait — shouldn't the output be 8192 not 28672?)
```

Actually, for column-parallel linear layers (like the Q/K/V projection or FFN gate):
- Input is broadcast (or replicated) to all ranks
- Each rank computes with its weight column shard
- No allreduce needed — outputs are already sharded

For row-parallel linear layers (like attention output projection or FFN down):
- Each rank has a partial sum of the result
- **Allreduce** combines them into the full result

**The allreduce fires at every row-parallel layer boundary** — typically:
- After attention output projection (once per layer)
- After FFN down projection (once per layer)
- Some models also after gate projection depending on architecture

For an 80-layer model (e.g. Llama-3 70B): ~160 allreduces per generated token, all on the critical path.

## What this notebook measures

1. A realistic decode loop: N layers, allreduce at each layer boundary
2. The communication budget as a fraction of total TPOT
3. Sensitivity to batch size and sequence length
4. How much CCL_WORKER_COUNT and algorithm selection affect the budget

:::{note}
All cells use the **launcher pattern**: MPI workload written to a temp file,
executed via `mpirun` as a subprocess.
:::

## 0. Environment Check

In [ ]:
import subprocess, os

def check(label, cmd, expect_in=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0 and (expect_in is None or expect_in in result.stdout + result.stderr)
    print(f"{'OK' if ok else 'FAIL'}  {label}")
    if not ok:
        print(f"     {result.stdout.strip()[:200] or result.stderr.strip()[:200]}")
    return ok

check("mpirun available",          "which mpirun")
check("Intel MPI loaded",          "mpirun --version", expect_in="Intel")
check("PyTorch installed",         "python -c 'import torch; print(torch.__version__)'")
check("oneCCL bindings installed", "python -c 'import oneccl_bindings_for_pytorch'")
check("XPU available",             "python -c 'import torch; assert torch.xpu.is_available()'")

## 1. TP Decode Loop — Single Model Config

This simulates Llama-3 70B decode with TP=4:
- 80 layers
- hidden_dim = 8192
- 2 row-parallel layers per transformer block (attn_out, ffn_down)
- batch=1, seq=1 (decode mode: one new token at a time)

:::{warning}
The benchmark cells below set `CCL_ALLREDUCE=ring` to isolate the ring algorithm on the
CPU path. For **production GPU inference**, leave `CCL_ALLREDUCE` unset so oneCCL uses
the `topo` algorithm (GPU-native scale-up). Setting any value other than `topo` causes
a GPU→CPU→GPU copy. Use `CCL_ALLREDUCE_SCALEOUT=ring` for scaleout control only.
:::

In [ ]:
%%writefile /tmp/ccl_tp_decode_loop.py
import os, time
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_ALLREDUCE"]     = "ring"  # NOTE: forces CPU path for benchmarking; omit for GPU inference
os.environ["CCL_PRIORITY"]      = "lifo"
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# === Model configuration ===
# Llama-3 70B: 80 layers, hidden=8192
NUM_LAYERS       = 80
HIDDEN_DIM       = 8192
ALLREDUCES_PER_LAYER = 2   # attn_out row-parallel + ffn_down row-parallel
BATCH_SIZE       = 1
SEQ_LEN          = 1       # decode: 1 new token per step
WARMUP_TOKENS    = 20
DECODE_TOKENS    = 200

# Allreduce message: after row-parallel, each rank has partial [batch, seq, hidden]
# In TP, the allreduce is over the full hidden dim (not sharded output)
msg_elements = BATCH_SIZE * SEQ_LEN * HIDDEN_DIM
msg_bytes    = msg_elements * 2  # BF16

if rank == 0:
    print(f"Config: Llama-3 70B | TP={world_size} | batch={BATCH_SIZE}")
    print(f"Allreduce message: {msg_bytes/1024:.1f} KB  ({msg_elements} BF16 elements)")
    print(f"Allreduces per token: {NUM_LAYERS * ALLREDUCES_PER_LAYER}")
    print(f"  ({NUM_LAYERS} layers × {ALLREDUCES_PER_LAYER} per layer)")
    print()

# Pre-allocate all buffers once — no allocation in the hot path
# Two buffers: one for attn_out, one for ffn_down (reused across layers)
attn_out_buf = torch.randn(msg_elements, dtype=torch.bfloat16, device=device)
ffn_down_buf = torch.randn(msg_elements, dtype=torch.bfloat16, device=device)

# Decode loop
token_comm_times = []

for tok in range(DECODE_TOKENS + WARMUP_TOKENS):
    dist.barrier()
    t_comm_start = time.perf_counter()

    for layer in range(NUM_LAYERS):
        # --- Attention output projection (row-parallel → allreduce) ---
        dist.all_reduce(attn_out_buf, op=dist.ReduceOp.SUM)

        # --- FFN down projection (row-parallel → allreduce) ---
        dist.all_reduce(ffn_down_buf, op=dist.ReduceOp.SUM)

    torch.xpu.synchronize(device)
    t_comm_end = time.perf_counter()

    if tok >= WARMUP_TOKENS:
        token_comm_times.append((t_comm_end - t_comm_start) * 1e3)  # ms

if rank == 0:
    token_comm_times.sort()
    total_ars = NUM_LAYERS * ALLREDUCES_PER_LAYER
    p50 = token_comm_times[len(token_comm_times)//2]
    p95 = token_comm_times[int(len(token_comm_times)*0.95)]
    p50_per_ar = p50 / total_ars * 1000  # microseconds

    print("=== TP Decode Communication Cost (comm only, no compute) ===")
    print(f"Allreduces per token         : {total_ars}")
    print(f"Total comm per token (p50)   : {p50:.3f} ms")
    print(f"Total comm per token (p95)   : {p95:.3f} ms")
    print(f"Per-allreduce latency (p50)  : {p50_per_ar:.1f} us")
    print()
    print("Comm as % of TPOT at various compute budgets:")
    for compute_ms in [1, 2, 5, 10, 20]:
        pct = p50 / (compute_ms + p50) * 100
        print(f"  compute={compute_ms:2d} ms/tok → comm overhead = {pct:.1f}%")

dist.destroy_process_group()

In [ ]:
import subprocess
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_tp_decode_loop.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 2. Batch Size Sensitivity

Decode often runs with batch sizes > 1 (continuous batching). As batch grows:
- Allreduce message size grows linearly with batch
- Compute time also grows (more tokens to process per step)
- The crossover point where comm matters decreases

This sweep shows how comm budget changes from batch=1 (pure latency mode) to batch=32.

In [ ]:
%%writefile /tmp/ccl_batch_sweep.py
import os, time, json
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_ALLREDUCE"]     = "ring"  # NOTE: forces CPU path for benchmarking; omit for GPU inference
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

NUM_LAYERS = 80
HIDDEN_DIM = 8192
ARS_PER_LAYER = 2
WARMUP = 5
ITERS  = 30

batch_sizes = [1, 2, 4, 8, 16, 32]
results = []

for batch in batch_sizes:
    msg_elems = batch * 1 * HIDDEN_DIM  # seq=1 for decode
    buf = torch.randn(msg_elems, dtype=torch.bfloat16, device=device)

    for _ in range(WARMUP):
        for _ in range(NUM_LAYERS * ARS_PER_LAYER):
            dist.all_reduce(buf, op=dist.ReduceOp.SUM)
        torch.xpu.synchronize(device)
    dist.barrier()

    times = []
    for _ in range(ITERS):
        t0 = time.perf_counter()
        for _ in range(NUM_LAYERS * ARS_PER_LAYER):
            dist.all_reduce(buf, op=dist.ReduceOp.SUM)
        torch.xpu.synchronize(device)
        times.append((time.perf_counter() - t0) * 1e3)
    dist.barrier()

    times.sort()
    if rank == 0:
        p50 = times[len(times)//2]
        p95 = times[int(len(times)*0.95)]
        results.append({"batch": batch, "msg_kb": msg_elems*2//1024,
                        "p50_ms": round(p50, 3), "p95_ms": round(p95, 3)})

if rank == 0:
    print(f"{'Batch':>6} {'Msg (KB)':>10} {'p50 comm/tok (ms)':>20} {'p95 comm/tok (ms)':>20}")
    print("-" * 60)
    for r in results:
        print(f"{r['batch']:>6} {r['msg_kb']:>10} {r['p50_ms']:>20.3f} {r['p95_ms']:>20.3f}")
    print()
    print(json.dumps(results))

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_batch_sweep.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 3. Async Overlap: Allreduce Concurrent with KV Cache Update

In a real decode server, while the allreduce is in flight, the CPU can:
- Update the KV cache index
- Do sampling bookkeeping
- Schedule the next kernel

Using `async_op=True` lets you fire the collective and do CPU work concurrently.
The XPU compute that depends on the allreduce result cannot start until `work.wait()`,
but CPU-side work (metadata, indices) can proceed.

:::{warning}
The "CPU overlap" in this demo is intentionally minimal (`kv_index` list writes), so the
async and sync patterns will show similar wall times. Real benefit only appears when
CPU-side work per layer takes a significant fraction of the allreduce latency (~20–50 µs
on BMG/CRI decode). The value of this pattern in production is not throughput improvement
but **latency hiding** — the CPU does useful bookkeeping instead of spinning on the sync.
If you see no improvement here, that is expected and correct.
:::

In [ ]:
%%writefile /tmp/ccl_decode_async.py
import os, time
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_ALLREDUCE"]     = "ring"  # NOTE: forces CPU path for benchmarking; omit for GPU inference
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

NUM_LAYERS = 80
HIDDEN_DIM = 8192
WARMUP = 10
ITERS  = 50

attn_buf = torch.randn(HIDDEN_DIM, dtype=torch.bfloat16, device=device)
ffn_buf  = torch.randn(HIDDEN_DIM, dtype=torch.bfloat16, device=device)

# Simulated KV cache index (CPU-side bookkeeping)
kv_index = list(range(2048))

# Pattern A: synchronous allreduce (baseline)
sync_times = []
for tok in range(ITERS + WARMUP):
    t0 = time.perf_counter()
    for layer in range(NUM_LAYERS):
        dist.all_reduce(attn_buf, op=dist.ReduceOp.SUM)  # blocks
        dist.all_reduce(ffn_buf,  op=dist.ReduceOp.SUM)  # blocks
    torch.xpu.synchronize(device)
    if tok >= WARMUP:
        sync_times.append((time.perf_counter() - t0) * 1e3)

dist.barrier()

# Pattern B: async allreduce with CPU overlap
async_times = []
for tok in range(ITERS + WARMUP):
    t0 = time.perf_counter()
    for layer in range(NUM_LAYERS):
        # Fire both allreduces async
        work_attn = dist.all_reduce(attn_buf, op=dist.ReduceOp.SUM, async_op=True)
        work_ffn  = dist.all_reduce(ffn_buf,  op=dist.ReduceOp.SUM, async_op=True)

        # CPU overlap: update KV index while comm is in flight
        # This is purely illustrative — real servers update page tables here
        kv_index[layer % len(kv_index)] = layer + tok

        work_attn.wait()
        work_ffn.wait()

    torch.xpu.synchronize(device)
    if tok >= WARMUP:
        async_times.append((time.perf_counter() - t0) * 1e3)

dist.barrier()

if rank == 0:
    sync_times.sort()
    async_times.sort()
    p50_sync  = sync_times[len(sync_times)//2]
    p50_async = async_times[len(async_times)//2]
    print("Sync allreduce  p50:", f"{p50_sync:.3f} ms/token")
    print("Async allreduce p50:", f"{p50_async:.3f} ms/token")
    improvement = (p50_sync - p50_async) / p50_sync * 100
    if improvement > 0:
        print(f"Async overhead reduction: {improvement:.1f}%")
    else:
        print(f"Async has overhead vs sync: {-improvement:.1f}% slower")
        print("(Expected for minimal CPU work — CPU bookkeeping must be substantial to benefit)")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_decode_async.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 4. Environment Variable Impact — Side-by-Side

A practical comparison of the CCL environment variables that matter most for the
TP decode hot path. Run this to quantify the impact on your specific hardware.

In [ ]:
%%writefile /tmp/ccl_decode_tuning.py
import os, time, sys
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

label = os.environ.get("BENCH_LABEL", "default")

dist.init_process_group(backend="ccl")
rank   = dist.get_rank()
device = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# 80 layers × 2 allreduces × 16 KB message (batch=1, hidden=8192)
NUM_LAYERS    = 80
ARS_PER_LAYER = 2
buf = torch.randn(8192, dtype=torch.bfloat16, device=device)

for _ in range(20):
    for _ in range(NUM_LAYERS * ARS_PER_LAYER):
        dist.all_reduce(buf, op=dist.ReduceOp.SUM)
    torch.xpu.synchronize(device)

times = []
for _ in range(50):
    t0 = time.perf_counter()
    for _ in range(NUM_LAYERS * ARS_PER_LAYER):
        dist.all_reduce(buf, op=dist.ReduceOp.SUM)
    torch.xpu.synchronize(device)
    times.append((time.perf_counter() - t0) * 1e3)

times.sort()
if rank == 0:
    print(f"{label:<50} p50={times[25]:.3f}ms  p95={times[47]:.3f}ms")

dist.destroy_process_group()

In [ ]:
import subprocess, os

configs = [
    {"BENCH_LABEL": "ofi + ring + 1 worker + lifo (recommended)",
     "CCL_ATL_TRANSPORT": "ofi", "CCL_ALLREDUCE": "ring",
     "CCL_WORKER_COUNT": "1", "CCL_PRIORITY": "lifo"},

    {"BENCH_LABEL": "ofi + ring + 1 worker (no priority)",
     "CCL_ATL_TRANSPORT": "ofi", "CCL_ALLREDUCE": "ring",
     "CCL_WORKER_COUNT": "1"},

    {"BENCH_LABEL": "mpi + ring + 1 worker",
     "CCL_ATL_TRANSPORT": "mpi", "CCL_ALLREDUCE": "ring",
     "CCL_WORKER_COUNT": "1"},

    {"BENCH_LABEL": "ofi + auto (CCL chooses algorithm)",
     "CCL_ATL_TRANSPORT": "ofi", "CCL_WORKER_COUNT": "1"},

    {"BENCH_LABEL": "ofi + ring + 2 workers (not recommended for GPU buffers)",
     "CCL_ATL_TRANSPORT": "ofi", "CCL_ALLREDUCE": "ring",
     "CCL_WORKER_COUNT": "2"},  # Intel docs advise CCL_WORKER_COUNT<=1 for GPU buffers
]

print(f"{'Config':<50} {'p50':>12} {'p95':>12}")
print("-" * 78)

for cfg in configs:
    env = {**os.environ, "CCL_LOG_LEVEL": "error", **cfg}
    r = subprocess.run(
        "mpirun -n 4 -ppn 4 python /tmp/ccl_decode_tuning.py",
        shell=True, capture_output=True, text=True, env=env
    )
    for line in r.stdout.strip().split("\n"):
        if line.strip():
            print(line)

## 5. Reading the Numbers: What This Means for Latency SLOs

The output above gives you the **pure communication cost** of TP decode. Use this table
to reason about your latency budget:

| TTFT/TPOT SLO | Target compute/tok | Max tolerable comm | Notes |
|---|---|---|---|
| 50ms TPOT | ~45ms | ~5ms | High bar — comm must be < 10% |
| 100ms TPOT | ~90ms | ~10ms | Comfortable |
| 200ms TPOT | ~185ms | ~15ms | Latency-insensitive workloads |

If your measured comm/token is higher than your tolerable budget:

1. **Check NUMA pinning first** — misaligned rings are the most common cause of 2-5× regressions
2. **Verify `ofi` transport** — mpi transport adds PMI overhead per collective
3. **Profile with VTune ITT** — `CCL_ITT_LEVEL=1` marks each collective in VTune
4. **Check PCIe bandwidth** — run `intel_gpu_top` and watch the PCIe BW during collective
5. **Consider TP degree** — TP=4 vs TP=8 doubles the comm cost for the same allreduce

## Summary

| Takeaway | Detail |
|---|---|
| **TP decode has 160 allreduces/token** | 80 layers × 2 row-parallel → all on critical path |
| **Batch=1 msgs are 16 KB** | Latency-bound; Ring (small-msg path) is the right algorithm |
| **Comm scales linearly with batch** | batch=32 → 512 KB msgs → bandwidth-bound |
| **Pre-allocate everything** | Alloc in hot path adds 10-50us variance |
| **ofi + ring + 1 worker is baseline** | Establish this first before any tuning |
| **NUMA pinning is the biggest lever** | Wrong pinning can 3-5× comm cost; check first |
| **KV cache goes to NIXL, not oneCCL** | Do not route bulk P→D transfers through allreduce |

**You have now seen the complete picture:**
- [Foundations](../chapters/00_foundations): Why collectives exist and how the algorithms work
- [Overview](../chapters/01_overview): The oneCCL API surface
- [Topology](../chapters/02_topology): Why Ring wins on NUMA hardware
- [GPU Hardware Constraints](../chapters/pvc_hardware_constraints): No GPU Direct DMA, no fabric — host staging required
- [When to Use](../chapters/03_when_to_use): Decision guide for each collective
- [Allreduce](03a_allreduce_walkthrough.ipynb): TP hot path, benchmarked
- [Allgather](03b_allgather.ipynb): Sequence parallelism KV gather
- [Alltoall](03c_alltoall_moe.ipynb): MoE expert dispatch and combine
- **This notebook**: End-to-end TP decode loop with real measurement
- [NIXL Boundary](../chapters/nixl_boundary.md): Where oneCCL ends and NIXL begins
- [Perf Tuning](../chapters/perf_tuning.md): Every CCL_* variable you need